In [ ]:
import torch

from occhio import ToyModel
from occhio.autoencoder import TiedLinearRelu
from occhio.distributions import CorrelatedPairs
from occhio.model_grid import ModelGrid, Axis, TrainingAxis
from occhio.visualization import (
    plot_embedding,
    plot_phase_change,
    plot_geometry,
    plot_feature_geometry,
    plot_feature_geometry_3d,
    plot_representation,
)

In [ ]:
device = "mps"

# Embeddings

In [ ]:
N_FEATURES = 10
N_HIDDEN = 2

In [ ]:
def create_model(params):
    generator = torch.Generator(device=device).manual_seed(199)

    return ToyModel(
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device, generator=generator),
        distribution=CorrelatedPairs(
            N_FEATURES,
            density=1 - params["Sparsity"],
            correlation=params["Correlation"],
            device=device,
            generator=generator,
        ),
        # Correlated pairs of diminishing importance, where within the second feature is also less important.
        importances=torch.tensor([1, 0.7] * int(N_FEATURES / 2))
        * (0.9 ** torch.arange(N_FEATURES)),
        device=device,
    )


grid = ModelGrid(
    create_model,
    axes=[
        Axis(label="Sparsity", values=[0.0, 0.8, 0.9, 0.99]),
        Axis(label="Correlation", values=[0.0, 0.5, 1]),
        # Axis(label="Importance", values=[1, 0.9, 0.7]),
    ],
)

In [ ]:
grid.fit(20000)

In [ ]:
plot_embedding(grid)

## Observations

- Like with the uniform distribution, sparsity enables superposition.
- There are different regimes:
    - The most important paris want to get dedicated directions, orthogonal to their correlated pair.
    - If sparsity increases it gets more crowded the additional pairs get represented in opposition at first and then a pentagon becomes a stable shape.
    - As correlation increases, the cost of interference does too and pairs may collapse into one single direction, in that case their norms tend to decrease as well (they sum to something close to 1, with the most important feature having a larger norm). The more correlation, the more equal representation each element of the pair gets.
    - As sparsity increases, correlated pairs are not represented orthogonally anymore, pairs start inferring with each other.
- The least important features get no representation.

# Transformations

In [ ]:
N_FEATURES = 20
N_HIDDEN = 5

In [ ]:
def create_model(params):
    generator = torch.Generator(device=device).manual_seed(199)

    return ToyModel(
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device, generator=generator),
        distribution=CorrelatedPairs(
            N_FEATURES,
            density=params["Density"],
            correlation=0.8,
            device=device,
            generator=generator,
        ),
        # Correlated pairs of diminishing importance, where within the second feature is also less important.
        importances=torch.tensor([1, 0.7] * int(N_FEATURES / 2))
        * (0.9 ** torch.arange(N_FEATURES)),
        device=device,
    )


grid = ModelGrid(
    create_model,
    axes=[Axis(label="Density", values=[1, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001])],
)

In [ ]:
grid.fit()

In [ ]:
plot_representation(grid)

# Phase Transitions

In [ ]:
N_FEATURES = 2
N_HIDDEN = 1
EXPERIMENT_SIZE = 10

In [ ]:
def create_model(params):
    generator = torch.Generator(device=device).manual_seed(199)

    return ToyModel(
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device, generator=generator),
        distribution=CorrelatedPairs(
            N_FEATURES,
            density=params["Density"],
            correlation=params["Correlation"],
            device=device,
            generator=generator,
        ),
        importances=params["Relative Importance"] ** torch.arange(N_FEATURES),
        device=device,
    )


grid = ModelGrid(
    create_model,
    axes=[
        Axis(
            label="Relative Importance", values=torch.logspace(-2, 2, EXPERIMENT_SIZE)
        ),
        Axis(label="Density", values=torch.logspace(-2, 0, EXPERIMENT_SIZE)),
        TrainingAxis(label="Correlation", values=torch.linspace(0, 0.01, 10)),
    ],
)

In [ ]:
grid.fit()

In [ ]:
plot_phase_change(grid, tracked_feature=1)

## Observations

We get a similar story to the uniform sparce distribution:
- With sufficient sparsity, the features live in superposition
- In a dense regime, the most important feature gets full representation
- What changes is that in the superposition regime is more continuous, the features are not antipodal but represented in the same direction, the more important feature gets a higher norm.
    - The higher the correlation the more interference is tolerated.
- Interestingly there is a sharp change from the superposition regime with no-correlation (antipodal pairs) to any small amount of correlation (much less superposition at first features share the same direction)
- Antipodal pairs still make sense for very low correlation with specific relative importance (~5)

# Geometry

In [ ]:
N_FEATURES = 200
N_HIDDEN = 16

In [ ]:
def create_model(params):
    generator = torch.Generator(device=device).manual_seed(199)

    return ToyModel(
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device, generator=generator),
        distribution=CorrelatedPairs(
            N_FEATURES,
            density=params["Density"],
            correlation=0.8,
            device=device,
            generator=generator,
        ),
        # importances=0.996 ** torch.arange(N_FEATURES),
        # Correlated pairs of diminishing importance, where within the second feature is also less important.
        importances=torch.tensor([1, 0.7] * int(N_FEATURES / 2))
        * (0.996 ** torch.arange(N_FEATURES)),
        device=device,
    )


grid = ModelGrid(
    create_model,
    axes=[
        Axis(label="Density", values=10 ** torch.linspace(0, -3, 50)),
    ],
)

In [ ]:
grid_history = grid.fit(20000, snapshot_interval=500)

In [ ]:
plot_geometry(grid)

In [ ]:
plot_feature_geometry([grid.models[16]])

In [ ]:
plot_feature_geometry_3d([grid.models[16]])

## Observations

- There is initial structure, correlated pairs want to organize themselves in paris, then in quads of 2 pairs
- Superstructures of these quads form in intermediate regimes
- Past a certain sparsity these structures collapse into much more entangled objects